<a href="https://colab.research.google.com/github/MichaelZimm20/xn-sitstayforever-semantic-attention/blob/main/notebooks/xn_sitstayforever_data_and_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
#--- DIRECTORIES-----
BASE_DIR = '/content/drive/MyDrive/xn-sitstayforever'
DATASET_DIR = f'{BASE_DIR}/datasets'
OUTPUTS_DIR = f'{BASE_DIR}/outputs'
CHECKPOINTS_DIR = f'{BASE_DIR}/checkpoints'
IMAGE_DIR = f'{DATASET_DIR}/product_images'

print(DATASET_DIR)

/content/drive/MyDrive/xn-sitstayforever/datasets


# Setup & Imports

In [ ]:
# Installs for missing dependencies not natively supported
!pip install open-clip-torch -q

print("Open CLIP install successful!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
Open CLIP install successful!


In [ ]:
#------- IMPORTS -------------
# --System Libraries---
import os
import sys
import random
from pathlib import Path

# --Data Handling---
import numpy as np
import pandas as pd
import requests

# --Visualizations---
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# --Preprocessing & Encoding---
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans



# --Image Processing---
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from tqdm import tqdm # wraps any loop and gives a live progress bar when loading the images in a loop

# --Deep Learning---
import torch
import torchvision.transforms as transforms

# --CLIP---
import open_clip

print("All Libraries have been successfully imported!")

All Libraries have been successfully imported!


# Dataset Loading

In [ ]:
import os

# File path smoke test, since files are in a google drive and mounted
# for item in os.listdir(BASE_DIR):
#   print(repr(item))
print('=' * 50)
for item in os.listdir(DATASET_DIR):
  print(item)
print('=' * 50)
# check exact folder name including hidden characters
items = os.listdir(BASE_DIR)
for item in items:
  print(repr(item))

# check it path exists
print(os.path.exists(DATASET_DIR))

SSF_CV_Dataset.xlsx
pet_cv_dataset_full.xlsx
product_images
'datasets'
'checkpoints'
'outputs'
True


In [ ]:
# ------ LOADING THE DATASET ------
''' The dataset consists of 54 images,
text, 2 spreadsheets which need to
be imported, analyzed and clean during
this preprocessing stage'''

# load the pet_cv_dataset.xlsx using pandas
df_images = pd.read_excel(f'{DATASET_DIR}/pet_cv_dataset_full.xlsx', sheet_name='images_cv_features')
df_products = pd.read_excel(f'{DATASET_DIR}/pet_cv_dataset_full.xlsx', sheet_name='product_summary')

# load SSF_CV_Dataset.xlsx using pandas
df_ssf = pd.read_excel(f'{DATASET_DIR}/SSF_CV_Dataset.xlsx', sheet_name='images_cv_features')
df_keywords = pd.read_excel(f'{DATASET_DIR}/SSF_CV_Dataset.xlsx', sheet_name='keywords_dataset')

# --Outputs---#
print(f'(df_images:   {df_images.shape}')
print(f'(df_products:   {df_products.shape}')
print(f'(df_ssf:   {df_ssf.shape}')
print(f'(df_keywords:   {df_keywords.shape}')

print('=' * 50)
''' Note:
df_images is outputting 145,44 instead of 54,44 to follow the 54 images. Its reading all 100 metadata rows plus the
54 rows combined. Will resolve this next section'''

(df_images:   (145, 44)
(df_products:   (100, 12)
(df_ssf:   (6, 36)
(df_keywords:   (146, 12)


' Note:\ndf_images is outputting 145,44 instead of 54,44 to follow the 54 images. Its reading all 100 metadata rows plus the\n54 rows combined. Will resolve this next section'

In [ ]:
# ------ LOADING THE IMAGES FROM GOOGLE DRIVE ------
''' This section will handle loading the images from the dataset. The images are stored in a folder on
Google drive

- since the imags can come in different channel formats, RGBA, grayscale, palette-based, it would be best to unify the formats
  - this will help our other models like CLIP, GradCAM etc to interpret the the same format instead of mixed formats



Storing the images and filepaths seperately for useablilty for other libaries and models. Better for memory and speed
'''

images = {} # store image objects in empty dictionary, ordered by filename
image_paths = {} # store full image file paths by filename to ensure we have correct location

# check file name, path, extension and sort
'''tqdm adds a real-time progress bar to the loop to track execution speed
    and time for image uploads to the dictionaries '''
for filename in tqdm(sorted(os.listdir(IMAGE_DIR))):
  if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
    filepath = os.path.join(IMAGE_DIR, filename)

    try:
      img = Image.open(filepath).convert('RGB') # read the image from drive and convert pixels to RGB (to standardize color 3 color channels)
      images[filename] = img
      image_paths[filename] = filepath
    except Exception as e:
      print(f"Error loading image {filename}: {e}")
      continue

print(f'\nLoaded {len(images)} images from {IMAGE_DIR}')


100%|██████████| 63/63 [00:01<00:00, 38.72it/s]


Loaded 63 images from /content/drive/MyDrive/xn-sitstayforever/datasets/product_images


In [ ]:
''' Issue 1: After filtering the rows, seems as if too many rows were removed. Specially the ones that are tier 'low'
Validating what happened

 '''

print("Before filtering:")
for tier, count in df_images['performance_tier'].value_counts().items():
    print(f"  {tier}: {count}")

print("Does low tier rows have brightness_mean values? :")
low_tier_rows = df_images[df_images['performance_tier'] == 'low']
if low_tier_rows['brightness_mean'].notna().any():
    count_with_brightness = low_tier_rows['brightness_mean'].notna().sum()
    print(f"  Yes, {count_with_brightness} rows have brightness_mean values.")
else:
    count_without_brightness = low_tier_rows['brightness_mean'].isna().sum()
    print(f"  No, {count_without_brightness} rows do not have brightness_mean values.")

Before filtering:
  high: 75
  medium: 33
  low: 31
  sponsor: 6
Does low tier rows have brightness_mean values? :
  No, 31 rows do not have brightness_mean values.


In [ ]:
# ---- Data Quality -----------------------

''' this section will be used to filter and clean the features and metadata

After troubleshooting and looking through the dataset spreadsheet again, found that the tier breakdown had missing images
and for those images brightness_mean' was NaN

Of the 145 rows in pet_cv_dataset_full
- 54 rows have computed CV features, where brightness_mean is not null = working images
- 91 rows are metadata-only = no downloaded images (Burts Bees and Bio Groom images)

Of the 54 usable image rows:
- high: 42 images (7 ASINs × 6 images)
- medium: 6 images (1 ASIN × 6 images)
- sponsor: 6 images (1 ASIN × 6 images — Sit Stay Forever)
- low: 0 images — no low tier products were downloaded


Conclusion/Impact:
With no low tier images, downstream model training will need to be adjusted.

treating it like a binary task (high v. medium/low)
OR
2-class problem (high v. sponsor benchmark)

will revisit in Notebook 2
'''

# filter to rows with computed CV features only (drop meta-data rows)
df_clean = df_images[df_images['brightness_mean'].notna()].copy()


# From starter template in xlsx. Convert CV Feature columns to numeric
CV_FEATURES = [
    'brightness_mean', 'contrast_std', 'sharpness_laplacian',
    'white_bg_pct', 'white_bg_compliance', 'product_dominance_score',
    'text_density_pct', 'clutter_score', 'color_warmth', 'saturation_mean',
    'symmetry_score', 'dominant_color_1_pct', 'ocr_word_count', 'ocr_keyword_count'

]


# create a new column for cleaned data and convert values of column to numeric data types
df_clean[CV_FEATURES] = df_clean[CV_FEATURES].apply(pd.to_numeric, errors='coerce') # turn any non-numeric string or invalid values to NaN

print(f'Total # of rows before filtering: {df_images.shape[0]}')
print(f'Total # of rows after filtering: {df_clean.shape[0]}')

# counting the number of values in the performance tier column then storing the label and count in a dictionary
print("~" * 50)
print("Total Tiers present:")
for tier, count in df_clean['performance_tier'].value_counts().items():
    print(f"  {tier}: {count}")

Total # of rows before filtering: 145
Total # of rows after filtering: 54
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Total Tiers present:
  high: 42
  sponsor: 6
  medium: 6


In [ ]:
# ---- Flagging problematic images ------------------
''' checking if images descriptions align with image instead stating Amazon UI text.
some images that were datascrapped, i found some images with incorrect context'''

# creating new column to mark images theat are good to go
df_clean['quality_flag'] = 'ok'

# --- Detection 1: Amazon Banner Screenshots ---------------
# detext is Amazon product images contain any Amazon UI text
# checking OCR for any promotional/navigational langugage not related to the product
banner_terms = [
    'spend', 'meds', 'prime', 'shop amazon',
    'school', 'get back', 'introducing', 'restrictions apply'
]

# checking if text is in banner terms
def is_banner_txt(ocr_text):
  if pd.isna(ocr_text):
    return False
  ocr_lower = str(ocr_text).lower()
  for term in banner_terms:
    if term in ocr_lower:
      return True
  return False

# running text in ocr_text to see if it contains the banner terms
banner_mask = df_clean['ocr_text'].apply(is_banner_txt)
df_clean.loc[banner_mask, 'quality_flag'] = 'banner_screenshot'

# --- Output # of Banner Screenshots detected -----------
print(f'Total # of banner screenshots: {banner_mask.sum()}')
if banner_mask.sum() > 0:
  # looking through flagged rows for verification
  print(df_clean.loc[banner_mask, ['asin', 'brand', 'image_type_label', 'ocr_text']].to_string(index=False))

# ---



Total # of banner screenshots: 6
      asin       brand image_type_label                                                                                  ocr_text
B01K4SH5JW Burt's Bees     product_main Spend less on your meds Prime members save up to 80%* amazon pharmacy *Restrictions apply
B01K4SH5JW Burt's Bees        lifestyle                                      Get back to school ready Our Top 100+ go-to supplies
B01K4SH5JW Burt's Bees      infographic                                          Introducing The Run Shop Gear to match your pace
B01K4SH5JW Burt's Bees        secondary           Shop early deals Shop Amazon devices up to /0% off prime day Limited-time offer
B01K4SH5JW Burt's Bees        secondary                                                         prime original New series prime A
B01K4SH5JW Burt's Bees        secondary                             Prime Day is = June 23-26 Members unlock early deals aa ——— >


In [ ]:
# ---- Flagging problematic images Continued... ------------------

# --- Detection 2: Low resolution images ----------------
''' Finding images below 400px. Upscaling these images to a consistent px could stretch the image and present more noise.
using the cleaned dataframe, checking columns width_px and height_px to see if they are below 400px

 '''
low_resolution_mask = (df_clean['width_px'] < 400) & (df_clean['height_px'] < 400)
df_clean.loc[low_resolution_mask, 'quality_flag'] = 'low_resolution'

# --- Output # of low res images detected -----------
print(f'Total # of low res images : {low_resolution_mask.sum()}')
if low_resolution_mask.sum() > 0:
  # looking through flagged rows for verification
  print(df_clean.loc[low_resolution_mask, ['asin', 'brand', 'width_px', 'height_px']].to_string(index=False))

Total # of low res images : 17
      asin        brand  width_px  height_px
B0002DIUVQ    Bio-Groom     230.0      230.0
B0002DIUVQ    Bio-Groom     115.0      115.0
B0002DIUVQ    Bio-Groom     345.0      345.0
B0002DIUVQ    Bio-Groom     230.0      230.0
B0002DIUVQ    Bio-Groom     115.0      115.0
B0002DIUVQ    Bio-Groom     345.0      345.0
B001GY0NPG Isle of Dogs     354.0      354.0
B001GY0NPG Isle of Dogs     354.0      354.0
B001GY0NPG Isle of Dogs     354.0      354.0
B001GY0NPG Isle of Dogs     354.0      354.0
B001GY0NPG Isle of Dogs     354.0      354.0
B001GY0NPG Isle of Dogs     354.0      354.0
B0002DINX6    Sulfodene     230.0      230.0
B0002DINX6    Sulfodene     115.0      115.0
B0002DINX6    Sulfodene     345.0      345.0
B0002DINX6    Sulfodene     230.0      230.0
B0002DINX6    Sulfodene     115.0      115.0


In [ ]:
#--- Binary Tier Relabeling -------
'''
This cell will handle the relabeling of the tiers do to no 'low' tier images and the six SSF 'sponsor' tier getting
relabeled for training purposes.

Setting up 2 main tiers 'high' versus 'medium' as of now and will address the imbalance in Notebook 2
'''

# preserve the original performance_tier column by making a copy
df_clean['tier_original'] = df_clean['performance_tier'].copy()

# relabel sponsor tier to high
df_clean['performance_tier'] = df_clean['performance_tier'].replace('sponsor', 'high')

#remove low tiers there is limited data and no downloads
df_clean = df_clean[df_clean['performance_tier'].isin(['high', 'medium'])]

def tier_counter():
  # count the number of items in each tier group
  for tier, count in df_clean['performance_tier'].value_counts().items():
    print(f"  {tier}: {count}")

print('=' * 50)
print(f'Total useable images: {len(df_clean)}')

Total useable images: 54


In [ ]:
#------ Data quality summary -----
print('Data Quality Summary')
print('=' * 55)

#total rows in raw datasets
print(f'Total Rows in raw dataset: {df_images.shape[0]}')

#total rows in cleaned dataset
print(f'Total Usable Rows after filtering the dataset: {df_clean.shape[0]}')

# number of metadata rows dropped
print(f'Metadata rows dropped: {df_images.shape[0] - df_clean.shape[0]}')

print('~' * 55)
print("Number of quality flags:")
for flag, count in df_clean['quality_flag'].value_counts().items():
  print(f'  {flag}: {count}')

print('~' * 55)
tier_counter() # Call the function without wrapping it in another print()

Data Quality Summary
Total Rows in raw dataset: 145
Total Usable Rows after filtering the dataset: 54
Metadata rows dropped: 91
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Number of quality flags:
  ok: 31
  low_resolution: 17
  banner_screenshot: 6
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  high: 48
  medium: 6
